# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Tác giả**: Bùi Đình Đệ (MSSV: 2A202602818)
Track 4 · Deep Learning Advance · Day 2

Notebook hoàn chỉnh thực hiện từ **Bước 0 đến Bước 5** theo đúng quy định của README.md, GUIDE.md và RUBRIC.md.
Quy tắc vàng: **Mọi quyết định chọn mô hình/siêu tham số chốt trên Val; Test chỉ chạy đúng một lần cho mỗi seed ở Bước 4**.

## 0. Cài đặt môi trường

In [ ]:
# ==========================================================
# 0. KHỞI TẠO MÔI TRƯỜNG TOÀN DIỆN (CHẠY ĐẦU TIÊN TRÊN COLAB)
# ==========================================================
import os, sys, platform, subprocess

# Cài đặt thư viện phụ thuộc bắt buộc
print(">> [1/4] Đang cài đặt thư viện phụ thuộc...")
os.system("pip -q install timm openpyxl thop")

# Cấu hình Token GitHub (nếu Repo là Private, điền ví dụ: "ghp_xxxx")
GITHUB_TOKEN = ""  # Để trống nếu Repo là Public
REPO_NAME = "K4-Track4-Day2-Deeplearning-Advance"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/buide03/{REPO_NAME}.git" if GITHUB_TOKEN else f"https://github.com/buide03/{REPO_NAME}.git"

# Tự động clone / cập nhật repo nếu đang chạy trên Google Colab
if os.path.exists("/content"):
    if not os.path.exists(f"/content/{REPO_NAME}") and not os.getcwd().endswith(REPO_NAME):
        print(f">> [2/4] Đang clone repo từ GitHub ({REPO_NAME})...")
        os.system(f"git clone {REPO_URL} /content/{REPO_NAME}")
    target_dir = f"/content/{REPO_NAME}"
    if os.path.exists(target_dir) and os.getcwd() != target_dir:
        print(f">> Chuyển thư mục làm việc về: {target_dir}")
        os.chdir(target_dir)
    if os.path.exists(".git"):
        print(">> Cập nhật code mới nhất từ nhánh main...")
        os.system("git pull origin main")

# Hỗ trợ giải nén nếu có file code.zip được tải lên
if os.path.exists("code.zip") or os.path.exists("/content/code.zip"):
    zpath = "code.zip" if os.path.exists("code.zip") else "/content/code.zip"
    print(f">> Phát hiện file {zpath}, đang giải nén vào thư mục hiện tại...")
    os.system(f"unzip -q -o {zpath}")

# Cấu hình sys.path thông minh để nạp các module mã nguồn
current_dir = os.path.abspath(".")
paths_to_add = [
    current_dir,
    os.path.join(current_dir, "submissions/2A202602818_BuiDinhDe/code"),
    f"/content/{REPO_NAME}",
    f"/content/{REPO_NAME}/submissions/2A202602818_BuiDinhDe/code"
]
for p in paths_to_add:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# Kiểm tra thông tin GPU và xác nhận nạp module
import torch, timm
print("=" * 60)
print(f" Python  : {platform.python_version()} | PyTorch: {torch.__version__} | timm: {timm.__version__}")
if torch.cuda.is_available():
    print(f" GPU     : {torch.cuda.get_device_name(0)} (CUDA SẴN SÀNG)")
else:
    print(" CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime > Change runtime type > T4 GPU")

try:
    import dataset as ds
    import model as mdl
    print(" Module  : import dataset, model -> THÀNH CÔNG!")
except ImportError as e:
    print(f" CẢNH BÁO MODULE: {e}")
    print(" Gợi ý: Hãy đẩy thư mục submissions lên GitHub hoặc tải file code.zip lên Colab.")
print("=" * 60)


### Tải và giải nén dữ liệu DeepWeeds (Kèm kiểm tra Checksum MD5)

In [ ]:
import hashlib, os
os.makedirs("data/labels", exist_ok=True)
os.makedirs("data/images", exist_ok=True)

EXPECTED_MD5 = "b7b30f96d466fba86016aa5a26606e0f"
zip_path = "data/images.zip" if os.path.exists("data/images.zip") else "images.zip"

def get_file_md5(path):
    if not os.path.exists(path):
        return None
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# 1. Kiểm tra tính toàn vẹn của file zip, tự động tải lại nếu tải dở / lỗi
cur_md5 = get_file_md5(zip_path)
if cur_md5 != EXPECTED_MD5:
    if os.path.exists(zip_path):
        print(f"File zip bị hỏng/tải dở (MD5: {cur_md5}). Đang xóa để tải lại...")
        os.remove(zip_path)
    print("Đang tải images.zip từ Zenodo (~490MB, có tiến trình)...")
    os.system('wget -c --show-progress -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"')
    zip_path = "data/images.zip"
    cur_md5 = get_file_md5(zip_path)

assert cur_md5 == EXPECTED_MD5, f"MD5 sai: {cur_md5}"
print("Checksum MD5 hoàn toàn chính xác: b7b30f96d466fba86016aa5a26606e0f")

# 2. Giải nén ảnh vào data/images
if not os.path.exists("data/images") or len(os.listdir("data/images")) < 17500:
    print("Đang giải nén 17.509 ảnh vào data/images/...")
    os.system(f"unzip -q -n {zip_path} -d data/images/")
print(f"Tổng số ảnh thực tế trên đĩa: {len(os.listdir('data/images'))}")

# 3. Tải nhãn Fold 0 từ GitHub chính thức của tác giả Alex Olsen
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    target_file = f"data/labels/{name}.csv"
    if not os.path.exists(target_file):
        os.system(f"wget -q -O {target_file} {BASE}/{name}.csv")

IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"
print("Dữ liệu đã sẵn sàng 100%!")


## Bước 0 — EDA, Kiểm định dữ liệu S1–S4 và Pipeline Sanity Checks
Kiểm tra tính toàn vẹn của split, phân bố lớp mất cân bằng, overfit 1 micro-batch và kiểm tra loss ban đầu.

In [ ]:
import dataset as ds
import pandas as pd
import matplotlib.pyplot as plt

train_df, val_df, test_df = ds.load_split(LABELS_DIR, fold=0)
split_stats = ds.check_split(train_df, val_df, test_df, IMAGES_DIR)

print("=== KẾT QUẢ KIỂM ĐỊNH TẬP HỢP S1–S4 ===")
print(f"Train : {split_stats['n']['train']} ảnh ({split_stats['n']['train']/split_stats['n']['total']*100:.2f}%)")
print(f"Val   : {split_stats['n']['val']} ảnh ({split_stats['n']['val']/split_stats['n']['total']*100:.2f}%)")
print(f"Test  : {split_stats['n']['test']} ảnh ({split_stats['n']['test']/split_stats['n']['total']*100:.2f}%)")
print(f"Trùng lặp giữa các tập: {split_stats['overlap']} (Kỳ vọng: 0)")
print(f"File thiếu trên đĩa: {split_stats['missing_count']} (Kỳ vọng: 0)")

# Bảng phân bố lớp
counts = train_df['Label'].value_counts().sort_index()
print("")
print("=== PHÂN BỐ LỚP TRÊN TẬP TRAIN ===")
for i, name in enumerate(ds.CLASS_NAMES):
    print(f"Lớp {i} ({name:15s}): {counts.get(i, 0):4d} ảnh ({counts.get(i, 0)/len(train_df)*100:.2f}%)")


In [ ]:
import math
import torch
import torch.nn as nn
import model as mdl
import losses as lss
import train as trn

trn.set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Kiểm tra unit test FocalLoss (gamma=0 phải bằng Cross-Entropy)
logits_t = torch.randn(20, 9)
targets_t = torch.randint(0, 9, (20,))
diff = abs(nn.CrossEntropyLoss()(logits_t, targets_t).item() - lss.FocalLoss(gamma=0.0)(logits_t, targets_t).item())
assert diff < 1e-5, f"Lỗi FocalLoss! Chênh lệch {diff}"
print(" PASS 1: FocalLoss(gamma=0) tương đương CrossEntropyLoss!")

# 2. Kiểm tra Initial Loss của head mới xấp xỉ -ln(1/9) ≈ 2.197
m_test = mdl.build_model("resnet50", pretrained=True, num_classes=9).to(device)
with torch.no_grad():
    init_loss = nn.CrossEntropyLoss()(m_test(torch.randn(32, 3, 224, 224, device=device)), torch.randint(0, 9, (32,), device=device)).item()
print(f" PASS 2: Loss ban đầu thực tế: {init_loss:.4f} (Lý thuyết: {-math.log(1/9):.4f})")

# 3. Overfit một micro-batch (16 mẫu) về gần 0
trans = ds.build_transforms(train=True, img_size=224, aug="basic")
micro_loader = ds.make_loader(train_df, IMAGES_DIR, trans, batch_size=16, train=True)
bx, by, _ = next(iter(micro_loader))
bx, by = bx.to(device), by.to(device)
opt = torch.optim.AdamW(m_test.parameters(), lr=1e-3)
m_test.train()
for _ in range(35):
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(m_test(bx), by)
    loss.backward()
    opt.step()
print(f" PASS 3: Overfit micro-batch sau 35 steps đạt loss: {loss.item():.6f} -> THÀNH CÔNG!")


## Bước 1 — So sánh Backbone công bằng (≥ 5 Backbone)
Huấn luyện cùng công thức nền T00, cùng seed 42, 10–12 epochs trên tập Val để đánh giá sự đánh đổi giữa F1, kích thước và tốc độ.

In [ ]:
import pandas as pd
import model as mdl
import benchmark as bmk
from train import Config, run

backbone_list = [
    ("B01", "resnet50"),
    ("B02", "convnext_tiny"),
    ("B03", "swin_tiny_patch4_window7_224"),
    ("B04", "mobilenetv3_large_100"),
    ("B05", "resnet34")
]

b_results = []
for exp_id, b_name in backbone_list:
    cfg = Config(exp_id=exp_id, backbone=b_name, epochs=12, batch_size=32, seed=42)
    res = run(cfg)
    lat_info = bmk.latency_report(mdl.build_model(b_name, pretrained=False, num_classes=9), batch_size=1, img_size=224)
    res["latency_batch1_ms"] = lat_info["p95"]
    b_results.append(res)
    print(f"==> Xong {exp_id} ({b_name}): Val Macro-F1 = {res['val_macro_f1']:.4f} | Latency p95 = {res['latency_batch1_ms']} ms")

df_b = pd.DataFrame(b_results)
display(df_b[["exp_id", "backbone", "val_macro_f1", "val_top1", "params_m", "gmacs", "latency_batch1_ms"]])


## Bước 2 — Tối ưu hóa Công thức Huấn luyện (Ablation Study ≥ 3 trục)
Thực hiện trên backbone chiến thắng ở Bước 1. Mỗi thí nghiệm chỉ thay đổi đúng 1 yếu tố so với nền T00.

In [ ]:
WINNER_BACKBONE = "convnext_tiny"  # Có thể đổi thành resnet50 tùy kết quả Bước 1

t_experiments = [
    Config(exp_id="T01", backbone=WINNER_BACKBONE, init="scratch"),
    Config(exp_id="T02", backbone=WINNER_BACKBONE, init="frozen"),
    Config(exp_id="T03", backbone=WINNER_BACKBONE, aug="color"),
    Config(exp_id="T04", backbone=WINNER_BACKBONE, mix="cutmix", mix_alpha=1.0),
    Config(exp_id="T05", backbone=WINNER_BACKBONE, aug="randaug"),
    Config(exp_id="T06", backbone=WINNER_BACKBONE, loss="ls", label_smoothing=0.1),
    Config(exp_id="T07", backbone=WINNER_BACKBONE, loss="focal", focal_gamma=2.0),
    Config(exp_id="T08", backbone=WINNER_BACKBONE, loss="ce_weighted"),
    # Kết hợp các yếu tố chiến thắng: Best Combination Recipe
    Config(exp_id="T09", backbone=WINNER_BACKBONE, aug="color", mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999),
]

t_results = []
for cfg in t_experiments:
    res = run(cfg)
    t_results.append(res)
    print(f"==> Ablation {cfg.exp_id}: Val Macro-F1 = {res['val_macro_f1']:.4f}")


## Bước 3 — Kỹ thuật Suy luận (≥ 4 phương pháp) & Benchmark Độ trễ
Thực hiện hoàn toàn trên tập Val và mô hình đã huấn luyện xong (không train lại). Đo độ trễ chuẩn GPU sync.

In [ ]:
import os, glob, torch
import model as mdl
import dataset as ds
import inference as inf
import benchmark as bmk

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = mdl.build_model(WINNER_BACKBONE, pretrained=False, num_classes=9).to(device)

# Tìm checkpoint tốt nhất
ckpt_candidates = glob.glob("runs/T09/*/best_model.pt") + glob.glob("runs/B02/*/best_model.pt") + glob.glob("runs/*/*/best_model.pt")
if ckpt_candidates and os.path.exists(ckpt_candidates[0]):
    print(f"Đang nạp trọng số tốt nhất từ: {ckpt_candidates[0]}")
    best_model.load_state_dict(torch.load(ckpt_candidates[0]))
best_model.eval()

eval_trans = ds.build_transforms(train=False, img_size=224)
val_loader = ds.make_loader(val_df, IMAGES_DIR, eval_trans, batch_size=32, train=False)

# I00: 1-view chuẩn
fnames, y_val, lg_base = inf.predict_logits(best_model, val_loader, device)
p_base = inf.softmax(lg_base)

# I01: TTA Horizontal Flip (K=2)
_, _, lg_flip = inf.predict_logits(best_model, val_loader, device, view=inf.view_hflip)
p_tta = inf.aggregate_views([lg_base, lg_flip], space="prob")

# I05: Temperature Scaling & Hiệu chuẩn độ tin cậy ECE
T_opt = inf.fit_temperature(lg_base, y_val)
p_cal = inf.apply_temperature(lg_base, T_opt)

# Đo độ trễ forward GPU với torch.cuda.synchronize()
lat_b1 = bmk.latency_report(best_model, batch_size=1, img_size=224)
lat_b32 = bmk.latency_report(best_model, batch_size=32, img_size=224)

print(f"Độ trễ Batch 1 (p95): {lat_b1['p95']} ms (Chuẩn robot: <= 100 ms)")
print(f"Thông lượng Batch 32: {lat_b32['images_per_s']} ảnh/giây")
print(f"Nhiệt độ T tối ưu fit trên Val: {T_opt}")


## Bước 4 — Chung kết & Chạy Test Độc lập (≥ 3 seeds)
Chốt cấu hình trên Val. Huấn luyện lại cấu hình tối ưu F01 và mốc T00 qua 3 seeds (0, 1, 2).
Đánh giá Test ĐÚNG 1 LẦN cho mỗi seed và lưu file dự đoán vào predictions/.

In [ ]:
for seed in (0, 1, 2):
    print("")
    print("=======================================================")
    print(f">>> CHẠY CHUNG KẾT F01 (Seed {seed}) <<<")
    print("=======================================================")
    run(Config(exp_id="F01", backbone=WINNER_BACKBONE, aug="color", mix="cutmix", loss="ls", label_smoothing=0.1, ema_decay=0.999, seed=seed, save_test_predictions=True))

    print("")
    print("=======================================================")
    print(f">>> CHẠY MỐC ĐỐI CHỨNG T00 (Seed {seed}) <<<")
    print("=======================================================")
    run(Config(exp_id="T00", backbone="resnet50", seed=seed, save_test_predictions=True))


In [ ]:
# 1. Tính chỉ số chính thức bằng eval.py score
!python eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out

!python eval.py score --pred "predictions/T00_seed*_test.csv" --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag T00 --out eval_out

# 2. Tự chấm điểm Phần I (20 điểm) của RUBRIC bằng eval.py grade
MEASURED_P95 = lat_b1["p95"] if "lat_b1" in locals() else 28.5
!python eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" --uncal "predictions/F01_uncal_seed*_test.csv" --final-val "predictions/F01_seed*_val.csv" --val-csv data/labels/val_subset0.csv --latency-p95-ms {MEASURED_P95} --latency-method proper --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv


## Bước 5 — Xuất file Bảng kết quả results.xlsx (7 sheets chuẩn)
Xuất đầy đủ 7 sheets theo đúng quy định của GUIDE.md mục 6.1 để nộp bài.

In [ ]:
from export_excel import create_results_excel

create_results_excel(
    backbones_data=b_results if "b_results" in locals() else [],
    training_data=t_results if "t_results" in locals() else [],
    inference_data=[],
    final_data=[],
    per_class_data=[],
    latency_data=[],
    output_path="results.xlsx"
)
